In [9]:
import itk
import numpy as np
from pathlib import Path
from typing import Union
import os
from PIL import Image
import numpy as np
from PIL import Image

In [17]:
def compare_pixels(inbuf):
    """Compare two pixels and return them in descending order."""
    if inbuf[0] > inbuf[1]:
        return [inbuf[0], inbuf[1]]
    else:
        return [inbuf[1], inbuf[0]]

def compare_stage(inbuf):
    """Compare pixels in pairs and sort them."""
    step = 2
    numpixels = len(inbuf)
    outbuf = inbuf.copy()
    
    for ii in range(0, numpixels - 1, step):
        t = compare_pixels([inbuf[ii], inbuf[ii + 1]])
        outbuf[ii] = t[0]
        outbuf[ii + 1] = t[1]
    
    return outbuf

def compare_stage1(inbuf):
    """Stage 1 comparison for odd indices."""
    numpixels = len(inbuf)
    tbuf = compare_stage(inbuf[0:numpixels-1])
    outbuf = np.append(tbuf, inbuf[numpixels-1])
    return outbuf

def compare_stage2(inbuf):
    """Stage 2 comparison for even indices."""
    numpixels = len(inbuf)
    tbuf = compare_stage(inbuf[1:numpixels])
    outbuf = np.append(inbuf[0], tbuf)
    return outbuf

def get_median_1d(inbuf):
    """Perform 1D median computation using sorting network approach."""
    numpixels = len(inbuf)
    tbuf = inbuf.copy()
    
    for ii in range(1, numpixels + 1):
        if ii % 2 == 1:  # Odd stage
            tbuf = compare_stage1(tbuf)
        else:  # Even stage
            tbuf = compare_stage2(tbuf)
    
    return tbuf

def get_median_2d(inbuf):
    """Perform 2D median computation by applying 1D median to rows and columns."""
    outbuf = inbuf.copy()
    nrows, ncols = outbuf.shape
    
    # Process each column
    for ii in range(ncols):
        col_data = outbuf[:, ii]
        col_data_out = get_median_1d(col_data)
        outbuf[:, ii] = col_data_out
    
    # Process each row
    for ii in range(nrows):
        row_data = outbuf[ii, :]
        row_data_out = get_median_1d(row_data)
        outbuf[ii, :] = row_data_out
    
    return outbuf

def getMinMaxMed_1d(inbuf):
    """Get min, median, and max from a 1D sorted buffer."""
    max_val = inbuf[0]
    med_val = inbuf[len(inbuf) // 2]
    min_val = inbuf[-1]
    
    return min_val, med_val, max_val

def getMinMaxMed_2d(inbuf):
    """Get min, median, and max from a 2D sorted buffer."""
    nrows, ncols = inbuf.shape
    max_val = inbuf[0, 0]
    med_val = inbuf[nrows // 2, ncols // 2]
    min_val = inbuf[-1, -1]
    
    return min_val, med_val, max_val

def get_center_data(min_val, med_val, max_val, center_data):
    """Determine whether to keep center data or replace with median."""
    if center_data <= min_val or center_data >= max_val:
        return med_val
    else:
        return center_data

def get_new_pixel(min3, med3, max3, min5, med5, max5, min7, med7, max7, min9, med9, max9, center_data):
    """Determine the new pixel value based on various window statistics."""
    if med3 > min3 and med3 < max3:
        new_pixel = get_center_data(min3, med3, max3, center_data)
    elif med5 > min5 and med5 < max5:
        new_pixel = get_center_data(min5, med5, max5, center_data)
    elif med7 > min7 and med7 < max7:
        new_pixel = get_center_data(min7, med7, max7, center_data)
    elif med9 > min9 and med9 < max9:
        new_pixel = get_center_data(min9, med9, max9, center_data)
    else:
        new_pixel = center_data
    
    return new_pixel

def adaptive_median_filter(image_path, output_path):
    """Apply the adaptive median filter to an image."""
    # Load the image and convert to grayscale
    img = Image.open(image_path).convert('L')
    I = np.array(img, dtype=np.float64)
    
    # Create output array of same size
    J = np.zeros_like(I)
    
    # Filter parameters
    smax = 9
    cp = smax // 2  # center pixel index
    
    # Define window indices
    w3 = np.arange(-1, 2)
    w5 = np.arange(-2, 3)
    w7 = np.arange(-3, 4)
    w9 = np.arange(-4, 5)
    
    # Calculate window ranges
    r3 = cp + w3  # 3x3 window
    r5 = cp + w5  # 5x5 window
    r7 = cp + w7  # 7x7 window
    r9 = cp + w9  # 9x9 window
    
    # Create a sliding window buffer
    window = np.zeros((smax, smax))
    
    nrows, ncols = I.shape
    
    # Process each column of the image
    datavalid = False
    for ii in range(ncols):
        if ii < ncols - smax + 1:  # Make sure we have enough columns
            for jj in range(nrows - smax + 1):  # Make sure we have enough rows
                # Get the current column of data
                c_data = I[jj:jj+smax, ii]
                c_idx = ii
                
                # Shift window data to the right by one column
                window[:, 1:] = window[:, :-1]
                window[:, 0] = c_data
                
                # Get the different window regions
                d3x3 = window[np.ix_(r3, r3)]
                d5x5 = window[np.ix_(r5, r5)]
                d7x7 = window[np.ix_(r7, r7)]
                d9x9 = window[np.ix_(r9, r9)]
                
                center_pixel = window[cp, cp]
                
                # Apply median filtering to each window
                outbuf = get_median_1d(d3x3.flatten())
                min3, med3, max3 = getMinMaxMed_1d(outbuf)
                
                outbuf = get_median_2d(d5x5)
                min5, med5, max5 = getMinMaxMed_2d(outbuf)
                
                outbuf = get_median_2d(d7x7)
                min7, med7, max7 = getMinMaxMed_2d(outbuf)
                
                outbuf = get_median_2d(d9x9)
                min9, med9, max9 = getMinMaxMed_2d(outbuf)
                
                # Get the new pixel value
                pixel_val = get_new_pixel(
                    min3, med3, max3,
                    min5, med5, max5,
                    min7, med7, max7,
                    min9, med9, max9,
                    center_pixel
                )
                
                # Check if the output is valid (buffer has filled up)
                pixel_valid = datavalid
                datavalid = (c_idx >= smax - 1)
                
                # If valid, store the result
                if pixel_valid:
                    J[jj, ii] = pixel_val
    
    # Convert back to uint8 and save the filtered image
    filtered_img = Image.fromarray(J.astype(np.uint8))
    filtered_img.save(output_path)
    
    return I, J

In [18]:
BASE_PATH = Path(os.getcwd())
adaptive_median_filter(BASE_PATH / "inputs/salt_pepper2.png", "output.png")

(array([[255., 231., 130., ..., 252., 254., 255.],
        [255., 230., 123., ..., 251., 254., 255.],
        [255., 230., 124., ..., 251., 254., 255.],
        ...,
        [255., 240., 175., ..., 243., 252., 255.],
        [255., 240., 178., ..., 243., 252., 255.],
        [255., 241., 180., ..., 243., 252., 255.]], shape=(354, 361)),
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(354, 361)))